# 🧪 json2quantum Level 6
**Convierte un circuito cuántico definido con puertas básicas en JSON a un circuito Qiskit.**

**Niveles definidos en el Paper: Niveles de abstracción en QSE: una librería
de descomposición de operadores unitarios**

### Formato JSON esperado
```json
{
  "level": 6,
  "qubits": 4,
  "basis": ["CNOT", "Ry", "Rz", "X"],
  "operations": [
    {
      "gate": {
        "label": "elemental_rz",
        "name": "Rz",
        "target": 2,
        "parameter": 0.7854,
        "controls": []
      }
    }
  ]
}
```

## 📦 1. Instalación de dependencias

In [1]:
!pip install qiskit
!pip install pylatexenc
!pip install matplotlib

In [2]:
# Instalar Qiskit si no está disponible (necesario en Colab)
try:
    import qiskit
    print(f"✅ Qiskit {qiskit.__version__} ya instalado")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "qiskit", "qiskit-aer", "pylatexenc", "-q"])
    print("✅ Qiskit instalado correctamente")

✅ Qiskit 2.4.1 ya instalado


## 📚 2. Imports

In [3]:
import json
import math
from pathlib import Path
import matplotlib

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RZGate, RYGate, XGate, CXGate, HGate, ZGate, SGate, TGate
from qiskit.visualization import plot_histogram

print("✅ Imports OK")

✅ Imports OK


In [4]:
from qiskit import * 
qreg_q = QuantumRegister(4, 'q')
creg_c = ClassicalRegister(4, 'c')
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.rz(0.7854, qreg_q[2])
circuit.cx(qreg_q[0], qreg_q[2])

for gate_data in circuit.data:
    instruction, qubits, clbits = gate_data
    print(f"Gate: {instruction.name}, Qubits: {qubits}, Clbits: {clbits}")


Gate: rz, Qubits: [<Qubit register=(4, "q"), index=2>], Clbits: []
Gate: cx, Qubits: [<Qubit register=(4, "q"), index=0>, <Qubit register=(4, "q"), index=2>], Clbits: []


C:\Users\palob\AppData\Local\Temp\ipykernel_9556\2924057169.py:10: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  instruction, qubits, clbits = gate_data


In [5]:
circuit.data

[CircuitInstruction(operation=Instruction(name='rz', num_qubits=1, num_clbits=0, params=[0.7854]), qubits=(<Qubit register=(4, "q"), index=2>,), clbits=()), CircuitInstruction(operation=Instruction(name='cx', num_qubits=2, num_clbits=0, params=[]), qubits=(<Qubit register=(4, "q"), index=0>, <Qubit register=(4, "q"), index=2>), clbits=())]

## 🔧 3. Parser JSON → Qiskit

Mapeamos cada `name` del JSON a la compuerta Qiskit correspondiente.
Las compuertas parametrizadas (Rz, Ry) toman el campo `parameter` en radianes.
Las compuertas controladas (CNOT) usan el campo `controls`.

In [6]:
GATE_MAP: dict = {
    "X":    lambda p: XGate(),
    "Rz":   lambda p: RZGate(p),
    "Ry":   lambda p: RYGate(p),
    "CNOT": lambda p: CXGate()
}


def _resolve_control(c) -> int:
    if isinstance(c, int):
        return c
    if isinstance(c, dict):
        return c["qubit"]
    raise ValueError(f"Formato de control no reconocido: {c!r}")



def parse_json_to_circuit(json_data: dict) -> QuantumCircuit:
    n_qubits   = json_data["qubits"]
    operations = json_data["operations"]
    level      = json_data.get("level", "?")

    qr = QuantumRegister(n_qubits, name="q")
    cr = ClassicalRegister(n_qubits, name="c")
    qc = QuantumCircuit(qr, cr, name=f"nivel{level}")

    for op in operations:
        gate_def  = op["gate"]
        name      = gate_def["name"]
        parameter = gate_def.get("parameter", None)
        controls  = gate_def.get("controls", [])
        target    = gate_def["target"]

        gate = GATE_MAP[name](parameter).to_mutable()
        gate.label = gate_def.get("label", name)

        control_indices = [_resolve_control(c) for c in controls]
        all_qubits = [qr[c] for c in control_indices] + [qr[target]]
        qc.append(gate, all_qubits)

    #qc.measure(qr, cr)
    return qc



## 📂 4. Cargar el JSON y construir el circuito

In [13]:
# ── Cambia esta ruta al JSON que quieras parsear ──
JSON_FILE = "circuito.json"

with open(JSON_FILE, "r") as f:
    data = json.load(f)

print("📄 JSON cargado:")
print(json.dumps(data, indent=2))

📄 JSON cargado:
{
  "level": 6,
  "qubits": 4,
  "basis": [
    "CNOT",
    "Ry",
    "Rz",
    "X"
  ],
  "operations": [
    {
      "gate": {
        "name": "Rz",
        "target": 3,
        "parameter": 0.7854,
        "controls": []
      }
    },
    {
      "gate": {
        "name": "CNOT",
        "target": 2,
        "controls": [
          0
        ]
      }
    },
    {
      "gate": {
        "name": "Ry",
        "label": "Ry",
        "target": 1,
        "parameter": 0.7854,
        "controls": []
      }
    },
    {
      "gate": {
        "name": "CNOT",
        "label": "CNOT",
        "target": 1,
        "controls": [
          0
        ]
      }
    }
  ]
}


In [14]:
qc = parse_json_to_circuit(data)

## 🖼️ 5. Visualizar el circuito

In [15]:
print("\n🔭 Diagrama del circuito:")
qc.draw()


🔭 Diagrama del circuito:


CNOT                    CNOT 
q_0: ──────■───────────────────────■───
           │       ┌────────────┐┌─┴─┐ 
q_1: ──────┼───────┤ Ry(0.7854) ├┤ X ├─
         ┌─┴─┐     └────────────┘└───┘ 
q_2: ────┤ X ├─────────────────────────
     ┌───┴───┴────┐                    
q_3: ┤ Rz(0.7854) ├────────────────────
     └────────────┘                    
c: 4/══════════════════════════════════

---
## 🧩 8. Ejemplo: cargar desde string en lugar de archivo

Útil para pruebas rápidas sin necesitar un archivo en disco.

In [16]:
inline_json = """
{
    "level": 99,
    "qubits": 2,
    "basis": ["H", "CNOT"],
    "operations": [
        {
            "gate": {
                "label": "hadamard_q0",
                "name": "H",
                "target": 0,
                "controls": []
            }
        },
        {
            "gate": {
                "label": "entangle",
                "name": "CNOT",
                "target": 1,
                "controls": [{"qubit": 0}]
            }
        }
    ]
}
"""

data_bell = json.loads(inline_json)
qc_bell   = parse_json_to_circuit(data_bell)

qc_bell.draw()

KeyError: 'H'